In [3]:
import numpy as np
import tensorflow as tf

## --- STEP 1: Define the training data ---

In [4]:
# Input: [normalized_age, blue, red, green, sao_paulo, rio, curitiba]
tensor_normalized_people = np.array([
    [0.33, 1, 0, 0, 1, 0, 0],  # Erick
    [0,    0, 1, 0, 0, 1, 0],   # Ana
    [1,    0, 0, 1, 0, 0, 1],   # Carlos
])

In [17]:
# Output: Labels (one-hot encoded): [premium, medium, basic]
labels_names = ["premium", "medium", "basic"]
tensor_labels = np.array([
    [1, 0, 0],  # premium - Erick
    [0, 1, 0],  # medium  - Ana
    [0, 0, 1],  # basic   - Carlos
])

## --- STEP 2: Build and train the model ---

In [13]:
def train_model(input_xs, output_ys):
    model = tf.keras.Sequential([
        # First layer: 7 inputs, 80 neurons, ReLU activation
        tf.keras.layers.Dense(80, activation='relu', input_shape=(7,)),
        # Output layer: 3 neurons (one per category), softmax for probabilities
        tf.keras.layers.Dense(3, activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    # Train: 100 epochs, shuffle enabled, custom logging
    model.fit(
        input_xs,
        output_ys,
        epochs=100,
        shuffle=True,
        verbose=0,
        callbacks=[
            tf.keras.callbacks.LambdaCallback(
                on_epoch_end=lambda epoch, logs: print(
                    f"Epoch: {epoch}: loss = {logs['loss']}"
                )
            )
        ],
    )

    return model

## --- STEP 3: Predict ---

In [14]:
def predict(model, person):
    person_array = np.array(person)
    pred = model.predict(person_array, verbose=0)
    return [{"prob": prob, "index": i} for i, prob in enumerate(pred[0])]

## --- STEP 4: Run ---

In [15]:
print("Input:")
print(tensor_normalized_people)
print("Labels:")
print(tensor_labels)

Input:
[[0.33 1.   0.   0.   1.   0.   0.  ]
 [0.   0.   1.   0.   0.   1.   0.  ]
 [1.   0.   0.   1.   0.   0.   1.  ]]
Labels:
[[1 0 0]
 [0 1 0]
 [0 0 1]]


In [19]:
model = train_model(tensor_normalized_people, tensor_labels)

# New person to classify
tensor_normalized_person = [[0.2, 0, 1, 0, 0, 1, 0]]

/mnt/c/Users/Nelson/Documents/Code/Pós/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch: 0: loss = 1.2034744024276733
Epoch: 1: loss = 1.182668924331665
Epoch: 2: loss = 1.1621410846710205
Epoch: 3: loss = 1.141843557357788
Epoch: 4: loss = 1.121778130531311
Epoch: 5: loss = 1.1020232439041138
Epoch: 6: loss = 1.0824040174484253
Epoch: 7: loss = 1.063045620918274
Epoch: 8: loss = 1.0438851118087769
Epoch: 9: loss = 1.0249254703521729
Epoch: 10: loss = 1.0062516927719116
Epoch: 11: loss = 0.987825870513916
Epoch: 12: loss = 0.969637930393219
Epoch: 13: loss = 0.9516994953155518
Epoch: 14: loss = 0.9339943528175354
Epoch: 15: loss = 0.9165197014808655
Epoch: 16: loss = 0.899280846118927
Epoch: 17: loss = 0.8822857737541199
Epoch: 18: loss = 0.865685224533081
Epoch: 19: loss = 0.8493804931640625
Epoch: 20: loss = 0.8333763480186462
Epoch: 21: loss = 0.8176655173301697
Epoch: 22: loss = 0.8022070527076721
Epoch: 23: loss = 0.7870394587516785
Epoch: 24: loss = 0.7721520066261292
Epoch: 25: loss = 0.7574233412742615
Epoch: 26: loss = 0.742870032787323
Epoch: 27: loss = 0.

In [20]:
predictions = predict(model, tensor_normalized_person)
results = sorted(predictions, key=lambda p: p["prob"], reverse=True)
output = "\n".join(
    f"{labels_names[p['index']]} ({p['prob'] * 100:.2f}%)" for p in results
)
print(output)

medium (81.33%)
premium (11.21%)
basic (7.47%)
